In [10]:
# Part 1: imports and load
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

SUP, CTRY = "supplier_id", "supplier_country"
SEED = 42
df = pd.read_csv("data.csv")
print("Loaded:", df.shape)

Loaded: (113097, 18)


In [11]:
# Part 2: synthetic dates (recalibrated version)
rng = np.random.default_rng(SEED)
START = pd.Timestamp("2023-01-01")
n = len(df)
sup_quality = pd.Series(rng.beta(5, 2, df[SUP].nunique()), index=df[SUP].unique())
q = df[SUP].map(sup_quality).to_numpy()
ctry_transit = pd.Series(rng.integers(3, 26, df[CTRY].nunique()), index=df[CTRY].unique())

order_date = START + pd.to_timedelta(rng.integers(0, 730, n), unit="D")
planned_proc = rng.integers(1, 6, n)
planned_transit = np.clip(df[CTRY].map(ctry_transit).to_numpy() + rng.integers(-2, 3, n), 3, None)
expected_dispatch = order_date + pd.to_timedelta(planned_proc, unit="D")
expected_delivery = expected_dispatch + pd.to_timedelta(planned_transit, unit="D")

dispatch_delay = rng.poisson((1 - q) * 1.5)
transit_delay = rng.poisson((1 - q) * 1.5)
early = rng.binomial(1, 0.4 * q) * rng.integers(1, 3, n)
actual_dispatch = expected_dispatch + pd.to_timedelta(dispatch_delay, unit="D")
actual_delivery = actual_dispatch + pd.to_timedelta(planned_transit + transit_delay - early, unit="D")

df["order_date"] = order_date
df["dispatch_date"] = actual_dispatch
df["expected_delivery_date"] = expected_delivery
df["actual_delivery_date"] = actual_delivery
print(df[["order_date", "actual_delivery_date"]].head(3))

  order_date actual_delivery_date
0 2023-02-06           2023-02-23
1 2024-10-26           2024-11-03
2 2023-06-25           2023-07-13


In [12]:
# Part 3: shipment metrics and 0-1 scores
df = df.rename(columns={"delivery_time_deviation": "delivery_time_deviation_orig"})
df["lead_time_days_new"] = (df["actual_delivery_date"] - df["order_date"]).dt.days
df["expected_lead_time_days"] = (df["expected_delivery_date"] - df["order_date"]).dt.days
df["delivery_time_deviation"] = (df["actual_delivery_date"] - df["expected_delivery_date"]).dt.days
df["on_time"] = (df["delivery_time_deviation"] <= 0).astype(int)

# ASSUMPTION: both columns are 0-1 scores, higher = better
df["fulfil_score"] = df["order_fulfillment_status"].clip(0, 1)
df["cargo_score"] = df["cargo_condition_status"].clip(0, 1)
print("On-time rate:", round(df["on_time"].mean(), 3))

On-time rate: 0.579


In [13]:
# Part 4: supplier reliability score
g = df.groupby(SUP)
sup = pd.DataFrame({
    "avg_lead_time_days": g["lead_time_days_new"].mean(),
    "lead_time_std": g["lead_time_days_new"].std(),
    "avg_late_days": g["delivery_time_deviation"].apply(lambda x: x.clip(lower=0).mean()),
    "on_time_rate": g["on_time"].mean(),
    "fulfilment_perf": g["fulfil_score"].mean(),
    "cargo_perf": g["cargo_score"].mean(),
    "n_shipments": g.size(),
})
late = sup["avg_late_days"]
sup["delay_score"] = 1 - (late - late.min()) / (late.max() - late.min())
W = {"on_time_rate": 0.30, "delay_score": 0.20, "fulfilment_perf": 0.25, "cargo_perf": 0.25}
sup["supplier_reliability_score_new"] = 100 * sum(sup[c] * w for c, w in W.items())
sup.to_csv("supplier_scores.csv")
print("Reliability mean:", round(sup["supplier_reliability_score_new"].mean(), 1))

Reliability mean: 54.9


In [14]:
# Part 5: target, history features, time split, model
df = df.sort_values("order_date").reset_index(drop=True)
df["late_2d"] = (df["delivery_time_deviation"] >= 2).astype(int)
g = df.groupby(SUP)
df["hist_n"] = g.cumcount()
df["hist_on_time"] = g["on_time"].transform(lambda s: s.shift(1).expanding().mean())
df["hist_late_rate"] = g["late_2d"].transform(lambda s: s.shift(1).expanding().mean())
df["hist_mean_dev"] = g["delivery_time_deviation"].transform(lambda s: s.shift(1).expanding().mean())

feat = ["hist_n", "hist_on_time", "hist_late_rate", "hist_mean_dev",
        "expected_lead_time_days", "route_risk_level", "weather_condition_severity",
        "customs_clearance_time", "warehouse_inventory_level", "shipping_costs"]
X = df[feat].apply(pd.to_numeric, errors="coerce")
y = df["late_2d"]
cutoff = df["order_date"].quantile(0.7)
tr, te = df["order_date"] < cutoff, df["order_date"] >= cutoff

m = HistGradientBoostingClassifier(random_state=42).fit(X[tr], y[tr])
p = m.predict_proba(X[te])[:, 1]
print("Positive rate:", round(y.mean(), 3))
print("ROC-AUC:", round(roc_auc_score(y[te], p), 3),
      "| PR-AUC:", round(average_precision_score(y[te], p), 3))

Positive rate: 0.173
ROC-AUC: 0.753 | PR-AUC: 0.38


In [15]:
# Part 6: supplier-switch recommendations (training-period stats only)
train = df[tr]
s = train.groupby(SUP).agg(product_id=("product_id", "first"), late_rate=("late_2d", "mean"),
                           avg_lead=("lead_time_days_new", "mean"), n=("late_2d", "size"))
s = s[s["n"] >= 10]
best_alt = {}
for sid, r in s.iterrows():
    c = s[(s["product_id"] == r["product_id"]) & (s.index != sid)]
    if len(c):
        b = c.sort_values(["late_rate", "avg_lead"]).iloc[0]
        best_alt[sid] = (b.name, b["late_rate"], b["avg_lead"])

THR = 0.4
test = df[te].copy()
test["p_late"] = p
flag = test[test["p_late"] >= THR].copy()
flag["alt_supplier"] = flag[SUP].map(lambda x: best_alt.get(x, (np.nan,))[0])
flag["alt_late_rate"] = flag[SUP].map(lambda x: best_alt.get(x, (np.nan, np.nan))[1])
flag["recommend_switch"] = flag["alt_late_rate"].notna() & (flag["alt_late_rate"] < flag["p_late"] - 0.10)
print("Flagged:", len(flag), "| switch recommended:", flag["recommend_switch"].sum())

Flagged: 2615 | switch recommended: 2551


In [17]:
# Part 7 (fixed): export for the suggestions notebook
out = df.copy()
out["is_train"] = tr.to_numpy()
out["is_test"] = te.to_numpy()
out["p_late"] = np.nan
out.loc[te, "p_late"] = p
out["alt_supplier"] = flag["alt_supplier"].reindex(out.index)
out["recommend_switch"] = flag["recommend_switch"].reindex(out.index).fillna(False).astype(bool)
out.to_pickle("trustchain_state.pkl")
print("Saved:", out.shape, out["is_test"].sum(), out["p_late"].notna().sum())
print("Switch recommended:", out["recommend_switch"].sum(), "| rows with alt supplier:", out["alt_supplier"].notna().sum())

Saved: (113097, 38) 33933 33933
Switch recommended: 2551 | rows with alt supplier: 2615


In [18]:
# Cell 9: reroute, reallocate inventory, prioritise
THR = 0.4
rng2 = np.random.default_rng(7)

# --- Features from real columns (ASSUMPTION: higher = riskier for env columns, higher = more stock)
env_cols = ["route_risk_level", "weather_condition_severity", "customs_clearance_time"]
df["env_risk"] = df[env_cols].rank(pct=True).mean(axis=1)
df["inv_pct"] = df["warehouse_inventory_level"].rank(pct=True)
df["demand_pct"] = df["historical_demand"].rank(pct=True)
df["week"] = df["order_date"].dt.to_period("W").astype(str)
df["due_soon"] = 1 - df.groupby("week")["expected_delivery_date"].rank(pct=True)

# --- SYNTHETIC warehouses and transfer times (days)
N_WH = 6
df["warehouse"] = np.random.default_rng(11).integers(0, N_WH, len(df))
d = np.triu(rng2.integers(1, 6, (N_WH, N_WH)), 1)
dist = d + d.T                                   # symmetric, 0 on the diagonal

# Latest stock reading per warehouse from BEFORE each order date
last = df.groupby(["order_date", "warehouse"])["inv_pct"].last().unstack()
days = pd.date_range(df["order_date"].min(), df["order_date"].max())
stock = last.reindex(days).ffill().shift(1)

# --- SYNTHETIC alternative routes per country
countries = df[CTRY].unique()
alt = pd.DataFrame({
    "a1_t": rng2.uniform(.10, .20, len(countries)), "a1_c": rng2.uniform(.05, .15, len(countries)),
    "a2_t": rng2.uniform(.25, .40, len(countries)), "a2_c": rng2.uniform(.15, .30, len(countries)),
}, index=countries)
A1_RISK, A2_RISK = 0.7, 0.5                      # alt route keeps 70% / 50% of the env risk
ENV_THR, RESID_THR = 0.70, 0.55
SHORT_THR, DONOR_MIN, MAX_TRANSFER = 0.25, 0.60, 3

test = df[te].copy()
test["p_late"] = p
risky = test["p_late"] >= THR

# 1) Reroute
test["reroute_cond"] = test["env_risk"] >= ENV_THR
use_a1 = (test["env_risk"] * A1_RISK < RESID_THR).to_numpy()
a = alt.loc[test[CTRY]]
tm = np.where(use_a1, a["a1_t"].to_numpy(), a["a2_t"].to_numpy())
cm = np.where(use_a1, a["a1_c"].to_numpy(), a["a2_c"].to_numpy())
test["route_choice"] = np.where(test["reroute_cond"], np.where(use_a1, "alt1", "alt2"), "keep")
test["reroute_extra_days"] = np.where(test["reroute_cond"], np.ceil(test["expected_lead_time_days"] * tm), 0)
test["reroute_extra_cost"] = np.where(test["reroute_cond"], test["shipping_costs"] * cm, 0)

# 2) Reallocate inventory: nearest warehouse with enough stock
S = np.nan_to_num(stock.reindex(test["order_date"]).to_numpy(), nan=-1.0)
own = test["warehouse"].to_numpy()
S[np.arange(len(test)), own] = -1
t_cost = np.where(S >= DONOR_MIN, dist[own], 99)
test["donor_wh"] = t_cost.argmin(1)
test["transfer_days"] = t_cost.min(1)
test["realloc_cond"] = (test["inv_pct"] <= SHORT_THR) & (test["transfer_days"] <= MAX_TRANSFER)

# 3) Prioritise (within flagged orders)
test["p_rank"] = test["p_late"].rank(pct=True)
test["priority"] = 0.4 * test["demand_pct"] + 0.4 * test["p_rank"] + 0.2 * test["due_soon"]

ops = test[risky].copy()
ops["tier"] = pd.qcut(ops["priority"], [0, .7, .9, 1], labels=["P3", "P2", "P1"])
ops["rank_in_wh_week"] = ops.groupby(["week", "warehouse"])["priority"].rank(ascending=False, method="first")

# Combine with the supplier-switch result from Part 6 / Part 7
ops = ops.join(flag[["alt_supplier", "recommend_switch"]], how="left", rsuffix="_flag")
ops["recommend_switch"] = ops["recommend_switch"].fillna(False).astype(bool)
ops["n_actions"] = ops[["recommend_switch", "reroute_cond", "realloc_cond"]].sum(axis=1)

print("Flagged orders:", len(ops))
print(ops[["recommend_switch", "reroute_cond", "realloc_cond"]].sum())
print("Coverage (>=1 action):", round((ops["n_actions"] >= 1).mean(), 3))
print(ops["tier"].value_counts())
rr = ops[ops["reroute_cond"]]
print("Reroute: median extra days", rr["reroute_extra_days"].median(),
      "| median extra cost", round(rr["reroute_extra_cost"].median(), 2),
      "| route mix", rr["route_choice"].value_counts().to_dict())
print("Realloc: median transfer days", ops.loc[ops["realloc_cond"], "transfer_days"].median())

# Sensitivity to the probability threshold
for t in [0.3, 0.4, 0.5]:
    mk = test["p_late"] >= t
    print(f"thr={t}: flagged={mk.sum()}, reroute={(mk & test['reroute_cond']).sum()}, "
          f"realloc={(mk & test['realloc_cond']).sum()}")

Flagged orders: 2615
recommend_switch    2551
reroute_cond         284
realloc_cond         538
dtype: int64
Coverage (>=1 action): 0.979
tier
P3    1830
P2     523
P1     262
Name: count, dtype: int64
Reroute: median extra days 4.0 | median extra cost 52.36 | route mix {'alt1': 178, 'alt2': 106}
Realloc: median transfer days 2.0
thr=0.3: flagged=5045, reroute=567, realloc=926
thr=0.4: flagged=2615, reroute=284, realloc=538
thr=0.5: flagged=996, reroute=106, realloc=182
